In [1]:
## import 

import sys
# use line-buffering for both stdout and stderr
# sys.stdout = open(sys.stdout.fileno(), mode='w', buffering=1)
# sys.stderr = open(sys.stderr.fileno(), mode='w', buffering=1)

import hydra
from omegaconf import OmegaConf
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
import pathlib
import torch
import copy
import random
import wandb
import tqdm
import numpy as np
import shutil

from diffusion_policy.workspace.base_workspace import BaseWorkspace
from diffusion_policy.policy.robomimic_lowdim_policy import RobomimicLowdimPolicy
from diffusion_policy.dataset.base_dataset import BaseLowdimDataset
from diffusion_policy.env_runner.base_lowdim_runner import BaseLowdimRunner
from diffusion_policy.common.checkpoint_util import TopKCheckpointManager
from diffusion_policy.common.json_logger import JsonLogger
from diffusion_policy.common.pytorch_util import dict_apply, optimizer_to
from diffusion_policy.policy.robomimic_image_policy import RobomimicImagePolicy
from diffusion_policy.dataset.base_dataset import BaseImageDataset
from diffusion_policy.env_runner.base_image_runner import BaseImageRunner
from diffusion_policy.policy.diffusion_unet_hybrid_image_policy import DiffusionUnetHybridImagePolicy
from diffusion_policy.common.pytorch_util import dict_apply, optimizer_to
from diffusion_policy.model.diffusion.ema_model import EMAModel
from diffusion_policy.model.common.lr_scheduler import get_scheduler
from diffusion_policy.dataset.robomimic_replay_image_dataset import RobomimicReplayImageDataset
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, Sampler 
import datetime
import h5py

/home/carl_lab/miniconda3/envs/robodiff/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [ ]:
## Setup configuration

OmegaConf.register_new_resolver("eval", eval, replace=True)
config_path='diffusion_policy/config'
## should be in the diffusion_policy/config
config_name = "train_real_franka_pot"

with initialize(version_base=None, config_path=config_path):
    cfg_org = compose(
        config_name=config_name,
        overrides=[
            "hydra.run.dir=data/outputs/${now:%Y.%m.%d}/${now:%H.%M.%S}_${name}_${task_name}",
            "training.seed=42",
            "training.device=cuda:0"
        ],
    )
    print(cfg_org)
    
OmegaConf.resolve(cfg_org)

{'_target_': 'diffusion_policy.workspace.train_diffusion_unet_hybrid_workspace.TrainDiffusionUnetHybridWorkspace', 'shape_meta': {'action': {'shape': [10]}, 'obs': {'agentview_rgb': {'shape': [3, 240, 320], 'type': 'rgb'}, 'eye_in_hand_rgb': {'shape': [3, 240, 320], 'type': 'rgb'}, 'ee_states': {'shape': [16]}, 'joint_states': {'shape': [7]}, 'gripper_states': {'shape': [1]}}}, 'dataset_path': '/home/carl_lab/data_franka/60_drawer_bellpepper.hdf5', 'checkpoint': {'save_last_ckpt': True, 'save_last_snapshot': False, 'topk': {'format_str': 'epoch={epoch:04d}-test_mean_score={test_mean_score:.3f}.ckpt', 'k': 5, 'mode': 'max', 'monitor_key': 'test_mean_score'}}, 'dataloader': {'batch_size': 64, 'num_workers': 8, 'persistent_workers': False, 'pin_memory': True, 'shuffle': True}, 'val_dataloader': {'batch_size': 64, 'num_workers': 8, 'persistent_workers': False, 'pin_memory': True, 'shuffle': False}, 'dataset_obs_steps': 2, 'exp_name': 'default', 'horizon': 16, 'keypoint_visible_rate': 1.0, 

In [10]:
## Setup Workspace

class TrainDiffusionUnetHybridWorkspace(BaseWorkspace):
    include_keys = ['global_step', 'epoch']

    def __init__(self, cfg: OmegaConf, output_dir=None):
        super().__init__(cfg, output_dir=output_dir)

        # set seed
        seed = cfg.training.seed
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        # configure model
        self.model: DiffusionUnetHybridImagePolicy = hydra.utils.instantiate(cfg.policy)

        self.ema_model: DiffusionUnetHybridImagePolicy = None
        if cfg.training.use_ema:
            self.ema_model = copy.deepcopy(self.model)

        # configure training state
        self.optimizer = hydra.utils.instantiate(
            cfg.optimizer, params=self.model.parameters())

        # configure training state
        self.global_step = 0
        self.epoch = 0

timestamp = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
## location to save the checkpoints
output_dir = f"/home/carl_lab/ola/diffusion_policy/data/outputs/custom{timestamp}"
os.makedirs(output_dir, exist_ok=True)
print('output dir: ', output_dir)
workspace = TrainDiffusionUnetHybridWorkspace(cfg_org, output_dir=output_dir)

output dir:  /home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_19_29_50

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['joint_states', 'gripper_states', 'ee_states']
using obs modality: rgb with keys: ['agentview_rgb', 'eye_in_hand_rgb']
using obs modality: depth with keys: []
using obs modality: scan with keys: []
Diffusion params: 2.564722e+08
Vision params: 2.239418e+07


In [16]:
workspace.cfg.task.dataset.shape_meta

{'action': {'shape': [10]}, 'obs': {'agentview_rgb': {'shape': [3, 240, 320], 'type': 'rgb'}, 'eye_in_hand_rgb': {'shape': [3, 240, 320], 'type': 'rgb'}, 'ee_states': {'shape': [16]}, 'joint_states': {'shape': [7]}, 'gripper_states': {'shape': [1]}}}

In [ ]:
cfg = copy.deepcopy(workspace.cfg)

# resume training
if cfg.training.resume:
    lastest_ckpt_path = workspace.get_checkpoint_path()
    if lastest_ckpt_path.is_file():
        print(f"Resuming from checkpoint {lastest_ckpt_path}")
        workspace.load_checkpoint(path=lastest_ckpt_path)

new_config = OmegaConf.to_container(cfg.task.dataset, resolve=True )
del new_config['_target_'] ## remove from config

In [ ]:
## Setup Dataset

dataset = RobomimicReplayImageDataset(**new_config)
len(dataset)

cfg_dataloader = {key:value for key,value in cfg.dataloader.items()}

train_dataloader = DataLoader(dataset, **cfg_dataloader)
normalizer = dataset.get_normalizer()

# configure validation dataset
val_dataset = dataset.get_validation_dataset()
val_dataloader = DataLoader(val_dataset, **cfg.val_dataloader)

In [ ]:
## Testing 
batch = next(iter(train_dataloader))
batch.keys()
batch['action'].shape
batch['obs']['agentview_rgb'].shape

In [ ]:
workspace.model.set_normalizer(normalizer)
if cfg.training.use_ema:
    print("Using EMA")
    workspace.ema_model.set_normalizer(normalizer)

# configure lr scheduler
lr_scheduler = get_scheduler(
    cfg.training.lr_scheduler,
    optimizer=workspace.optimizer,
    num_warmup_steps=cfg.training.lr_warmup_steps,
    num_training_steps=(
        len(train_dataloader) * cfg.training.num_epochs) \
            // cfg.training.gradient_accumulate_every,
    # pytorch assumes stepping LRScheduler every epoch
    # however huggingface diffusers steps it every batch
    last_epoch=workspace.global_step-1
)

# configure ema
ema: EMAModel = None
if cfg.training.use_ema:
    ema = hydra.utils.instantiate(
        cfg.ema,
        model=workspace.ema_model)


In [ ]:
topk_manager = TopKCheckpointManager(
    save_dir=os.path.join(workspace.output_dir, 'checkpoints'),
    **cfg.checkpoint.topk
)

# device transfer
device = torch.device(cfg.training.device)
workspace.model.to(device)
if workspace.ema_model is not None:
    print("EMA is not none")
    workspace.ema_model.to(device)
optimizer_to(workspace.optimizer, device)